In [1]:
import pandas as pd
import joblib
import os
import sys

pd.set_option("display.max_columns", None)

In [2]:
sys.path.append(os.path.abspath("../preprocessing"))

from preprocess_churn import load_gold_data, prepare_ml_data

In [4]:
df = load_gold_data()

df.head()

,customer_id,churn,tenure,city_tier,warehouse_to_home,hour_spend_on_app,number_of_device_registered,satisfaction_score,number_of_address,complain,order_amount_hike_from_last_year,coupon_used,order_count,day_since_last_order,cashback_amount,preferred_login_device,preferred_payment_mode,gender,preferred_order_cat,marital_status,tenure_missing_flag,warehouse_to_home_missing_flag,hour_spend_on_app_missing_flag,day_since_last_order_missing_flag,is_new_customer,low_satisfaction_flag,has_complaint,inactive_customer_flag,high_cashback_customer_flag,batch_id,loaded_at,loaded_by
0,50001,1,4.0,3,6.0,3.0,3.0,2,9.0,1,11.0,1.0,1.0,5.0,160.0,mobile phone,debit card,female,laptop & accessory,single,0,0,0,0,1,1,1,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
1,50002,1,0.0,1,8.0,3.0,4.0,3,7.0,1,15.0,0.0,1.0,0.0,121.0,phone,upi,male,mobile,single,1,0,0,0,1,0,1,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
2,50003,1,0.0,1,30.0,2.0,4.0,3,6.0,1,14.0,0.0,1.0,3.0,120.0,phone,debit card,male,mobile,single,1,0,0,0,1,0,1,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
3,50004,1,0.0,3,15.0,2.0,4.0,5,8.0,0,23.0,0.0,1.0,3.0,134.0,phone,debit card,male,laptop & accessory,single,0,0,0,0,1,0,0,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user
4,50005,1,0.0,1,12.0,0.0,3.0,5,3.0,0,11.0,1.0,1.0,3.0,130.0,phone,cc,male,mobile,single,0,0,1,0,1,0,0,0,0,f190cdea-1610-40b4-8591-813dff94068a,2026-06-17 14:50:23.631095,local_user


In [5]:
customer_ids = df["customer_id"]

X, y, preprocessor, categorical_features, numeric_features = prepare_ml_data(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5630, 27)
y shape: (5630,)


In [6]:
selected_model = joblib.load("../../06_models/random_forest_churn_model.pkl")

selected_model

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
churn_probabilities = selected_model.predict_proba(X)[:, 1]
predicted_churn = selected_model.predict(X)

prediction_output = pd.DataFrame({
    "customer_id": customer_ids,
    "churn_probability": churn_probabilities,
    "predicted_churn": predicted_churn
})

prediction_output.head()

,customer_id,churn_probability,predicted_churn
0,50001,0.705,1
1,50002,0.955,1
2,50003,0.985,1
3,50004,0.910,1
4,50005,0.900,1


In [8]:
def assign_risk_level(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"


prediction_output["risk_level"] = prediction_output["churn_probability"].apply(assign_risk_level)

prediction_output.head()

,customer_id,churn_probability,predicted_churn,risk_level
0,50001,0.705,1,High Risk
1,50002,0.955,1,High Risk
2,50003,0.985,1,High Risk
3,50004,0.910,1,High Risk
4,50005,0.900,1,High Risk


In [9]:
prediction_output["risk_level"].value_counts()

risk_level
Low Risk       4680
High Risk       853
Medium Risk      97
Name: count, dtype: int64

In [10]:
prediction_output["risk_level"].value_counts(normalize=True) * 100

risk_level
Low Risk       83.126110
High Risk      15.150977
Medium Risk     1.722913
Name: proportion, dtype: float64

In [11]:
os.makedirs("../../05_outputs/predictions", exist_ok=True)

prediction_output.to_csv(
    "../../05_outputs/predictions/churn_predictions.csv",
    index=False
)

In [12]:
os.path.exists("../../05_outputs/predictions/churn_predictions.csv")

True

## Prediction Output Summary

The selected Random Forest model was used to generate churn predictions for all customers in the Gold table.

The output includes each customer's ID, predicted churn probability, predicted churn label, and churn risk level.

Risk levels were defined as follows:

- High Risk: churn probability >= 0.70
- Medium Risk: churn probability >= 0.40 and < 0.70
- Low Risk: churn probability < 0.40

The final prediction output was saved to `05_outputs/predictions/churn_predictions.csv`.